In [10]:
"""
Robustness Runner
=================
自动循环跑6个label配置，每个配置跑4个模型
最后汇总成robustness_summary.csv

文件夹结构:
D:\\phd\\during the GameStop\\4. robustness\\
├── run_robustness.py          ← 本文件
├── input\\
│   ├── gme_5min.csv
│   ├── Trade_halts_Historical.csv
│   ├── feature_groups_CLEAN.json
│   └── features\\
│       ├── cascade_features_5min.parquet
│       ├── network_features_5min.parquet
│       ├── temporal_features_5min.parquet
│       ├── text_features_5min.parquet
│       ├── burstiness_features_5min.parquet
│       ├── user_overlap_features_5min.parquet
│       └── text_duplication_features_5min.parquet
├── scripts\\
│   ├── clean_labels.ipynb
│   ├── Merge_all_features.ipynb
│   ├── xgboost.ipynb
│   ├── lightgbm.ipynb
│   ├── Logistic.ipynb
│   └── CNN-GRU.ipynb
└── results\\
    ├── config_base_10pct\\
    ├── config_low_5pct\\
    ...
    └── robustness_summary.csv
"""

import os
import sys
import shutil
import subprocess
import json
import pandas as pd
from pathlib import Path

# ============================================================
# 路径设置 — 只需要改这里
# ============================================================

BASE_DIR    = Path(r'D:\phd\during the GameStop\4. robustness')
INPUT_DIR   = BASE_DIR / 'input'
SCRIPTS_DIR = BASE_DIR / 'scripts'
RESULTS_DIR = BASE_DIR / 'results'

# scripts工作目录（nbconvert在这里执行）
WORK_DIR = BASE_DIR / 'workspace'

# ============================================================
# 6个label配置
# ============================================================

CONFIGS = [
    # (配置名,         LULD,   PRICE_Z, VOLUME_Z)
    ('base_10pct',   0.10,   2.0,     2.5),   # 当前主设置
    ('low_5pct',     0.05,   2.0,     2.5),   # 更宽松
    ('mid_7pct',     0.075,  2.0,     2.5),   # 中间
    ('strict_z',     0.10,   2.5,     3.0),   # 更严格z-score
    ('loose_z',      0.10,   1.5,     2.0),   # 更宽松z-score
    ('halt_only',    0.10,   99.0,    99.0),  # 只用官方halt
]

# 4个模型
MODELS = [
    'xgboost.ipynb',
    'lightgbm.ipynb',
    'Logistic.ipynb',
    'CNN-GRU.ipynb',
]

# ============================================================
# 工具函数
# ============================================================

def run_notebook(nb_path, work_dir):
    """用nbconvert原地执行notebook"""
    print(f'    ▶ 执行: {nb_path.name}')
    env = os.environ.copy()
    env['PYTHONIOENCODING'] = 'utf-8'
    env['PYTHONUTF8'] = '1'
    result = subprocess.run(
        [
            'jupyter', 'nbconvert',
            '--to', 'notebook',
            '--execute',
            '--inplace',
            '--ExecutePreprocessor.timeout=3600',  # 1小时超时
            str(nb_path)
        ],
        cwd=str(work_dir),
        capture_output=True,
        text=True,
        encoding='utf-8',
        env=env
    )
    if result.returncode != 0:
        print(f'    ✗ 错误: {result.stderr[-500:]}')
        raise RuntimeError(f'{nb_path.name} 执行失败')
    print(f'    ✓ 完成: {nb_path.name}')


def patch_notebook_params(src_nb, dst_nb, replacements):
    """
    修改notebook里的参数值
    replacements = [('OLD_LINE', 'NEW_LINE'), ...]
    """
    with open(src_nb, encoding='utf-8') as f:
        nb = json.load(f)

    for cell in nb['cells']:
        if cell['cell_type'] != 'code':
            continue
        src = ''.join(cell['source'])
        for old, new in replacements:
            src = src.replace(old, new)
        cell['source'] = [src]

    with open(dst_nb, 'w', encoding='utf-8') as f:
        json.dump(nb, f, indent=1, ensure_ascii=False)


def collect_results(work_dir, config_name, model_name):
    """从模型results文件夹收集PR-AUC等指标"""
    MODEL_RESULTS_MAP = {
        'xgboost.ipynb':  'results_xgb_EWS_15min',
        'lightgbm.ipynb': 'results_lgb_EWS_15min',
        'Logistic.ipynb': 'results_lr_EWS_15min',
        'CNN-GRU.ipynb':  'results_cnn_gru_EWS_15min',
    }

    folder = MODEL_RESULTS_MAP.get(model_name)
    if folder is None:
        print(f'    ⚠ 未知模型名: {model_name}')
        return None

    csv_path = work_dir / folder / 'ablation_results.csv'

    if csv_path.exists():
        df = pd.read_csv(csv_path, encoding='utf-8')
        df['config'] = config_name
        df['model']  = model_name.replace('.ipynb', '')
        print(f'    ✓ 收集结果: {folder}')
        return df

    print(f'    ⚠ 找不到: {csv_path}')
    existing = [p.name for p in work_dir.iterdir() if p.is_dir() and 'results' in p.name.lower()]
    if existing:
        print(f'    实际存在的results文件夹: {existing}')
    return None


def setup_workspace(config_name, luld, price_z, vol_z):
    """
    为每个配置建立独立工作目录
    复制scripts + input文件进去，修改参数
    """
    work = WORK_DIR / config_name
    work.mkdir(parents=True, exist_ok=True)

    # 建data子目录
    (work / 'data').mkdir(exist_ok=True)

    # 复制feature parquet到workspace
    feat_src = INPUT_DIR / 'features'
    for f in feat_src.glob('*.parquet'):
        shutil.copy2(f, work / f.name)

    # 复制feature_groups_CLEAN.json到workspace/data/
    shutil.copy2(INPUT_DIR / 'feature_groups_CLEAN.json', work / 'data' / 'feature_groups_CLEAN.json')

    # 复制market数据
    shutil.copy2(INPUT_DIR / 'gme_5min.csv',                work / 'gme_5min.csv')
    shutil.copy2(INPUT_DIR / 'Trade_halts_Historical.csv',   work / 'Trade_halts_Historical.csv')

    # ── 修改clean_labels参数 ──────────────────────────────────────────────────
    patch_notebook_params(
        src_nb = SCRIPTS_DIR / 'clean_labels.ipynb',
        dst_nb = work / 'clean_labels.ipynb',
        replacements=[
            ('LULD_THRESHOLD = 0.10',    f'LULD_THRESHOLD = {luld}'),
            ('PRICE_Z_THRESHOLD = 2.0',  f'PRICE_Z_THRESHOLD = {price_z}'),
            ('VOLUME_Z_THRESHOLD = 2.5', f'VOLUME_Z_THRESHOLD = {vol_z}'),
        ]
    )

    # ── 复制Merge_all_features（不需要改参数）────────────────────────────────
    shutil.copy2(SCRIPTS_DIR / 'Merge_all_features.ipynb', work / 'Merge_all_features.ipynb')

    # ── 复制模型脚本（不需要改参数，TARGET已经是EWS_15min）──────────────────
    for model in MODELS:
        shutil.copy2(SCRIPTS_DIR / model, work / model)

    return work


# ============================================================
# 主流程
# ============================================================

def main():
    print('\n' + '='*65)
    print('  ROBUSTNESS RUNNER')
    print('  6 label configs × 4 models = 24 runs')
    print('='*65)

    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    WORK_DIR.mkdir(parents=True, exist_ok=True)

    all_results = []

    for config_name, luld, price_z, vol_z in CONFIGS:
        print(f'\n{"─"*65}')
        print(f'  CONFIG: {config_name}')
        print(f'  LULD={luld}  PRICE_Z={price_z}  VOLUME_Z={vol_z}')
        print(f'{"─"*65}')

        # ── Step 1: 建workspace，修改参数 ─────────────────────────────────
        print('\n  Step 1: 准备workspace...')
        work = setup_workspace(config_name, luld, price_z, vol_z)
        print(f'    ✓ {work}')

        # ── Step 2: 跑clean_labels ────────────────────────────────────────
        print('\n  Step 2: 生成EWS labels...')
        run_notebook(work / 'clean_labels.ipynb', work)

        # ── Step 3: 跑Merge_all_features ──────────────────────────────────
        print('\n  Step 3: Merge特征...')
        run_notebook(work / 'Merge_all_features.ipynb', work)

        # ── Step 4: 跑四个模型 ────────────────────────────────────────────
        print('\n  Step 4: 跑模型...')
        for model in MODELS:
            print(f'\n    [{model}]')
            try:
                run_notebook(work / model, work)
                df = collect_results(work, config_name, model)
                if df is not None:
                    all_results.append(df)
            except RuntimeError as e:
                print(f'    ✗ 跳过: {e}')

        # ── Step 5: 复制results到config文件夹 ────────────────────────────
        config_out = RESULTS_DIR / f'config_{config_name}'
        if config_out.exists():
            shutil.rmtree(config_out)
        shutil.copytree(work, config_out,
                        ignore=shutil.ignore_patterns('*.parquet', '*.csv', '*.ipynb'))

        print(f'\n  ✓ Config {config_name} 完成')

    # ── Step 6: 汇总所有结果 ──────────────────────────────────────────────
    print('\n' + '='*65)
    print('  汇总结果...')

    if all_results:
        summary = pd.concat(all_results, ignore_index=True)

        # 加incremental PR-AUC（在每个config+model组内计算）
        summary = summary.sort_values(['config', 'model', 'experiment'])
        summary['incremental_pr_auc'] = (
            summary.groupby(['config', 'model'])['pr_auc']
            .diff()
            .fillna(0)
        )

        out_csv = RESULTS_DIR / 'robustness_summary.csv'
        summary.to_csv(out_csv, index=False)
        print(f'  ✓ 汇总表: {out_csv}')
        print(f'  总行数: {len(summary)}')

        # 打印关键对比
        print('\n  [full_with_coordination PR-AUC 对比]')
        full = summary[summary['experiment'] == 'full_with_coordination']
        pivot = full.pivot_table(
            index='config', columns='model', values='pr_auc'
        ).round(4)
        print(pivot.to_string())
    else:
        print('  ✗ 没有收集到任何结果')

    print('\n' + '='*65)
    print('  ROBUSTNESS RUNNER 完成')
    print('='*65)


if __name__ == '__main__':
    main()


  ROBUSTNESS RUNNER
  6 label configs × 4 models = 24 runs

─────────────────────────────────────────────────────────────────
  CONFIG: base_10pct
  LULD=0.1  PRICE_Z=2.0  VOLUME_Z=2.5
─────────────────────────────────────────────────────────────────

  Step 1: 准备workspace...
    ✓ D:\phd\during the GameStop\4. robustness\workspace\base_10pct

  Step 2: 生成EWS labels...
    ▶ 执行: clean_labels.ipynb
    ✓ 完成: clean_labels.ipynb

  Step 3: Merge特征...
    ▶ 执行: Merge_all_features.ipynb
    ✓ 完成: Merge_all_features.ipynb

  Step 4: 跑模型...

    [xgboost.ipynb]
    ▶ 执行: xgboost.ipynb
    ✓ 完成: xgboost.ipynb
    ✓ 收集结果: results_xgb_EWS_15min

    [lightgbm.ipynb]
    ▶ 执行: lightgbm.ipynb
    ✓ 完成: lightgbm.ipynb
    ✓ 收集结果: results_lgb_EWS_15min

    [Logistic.ipynb]
    ▶ 执行: Logistic.ipynb
    ✓ 完成: Logistic.ipynb
    ✓ 收集结果: results_lr_EWS_15min

    [CNN-GRU.ipynb]
    ▶ 执行: CNN-GRU.ipynb
    ✓ 完成: CNN-GRU.ipynb
    ✓ 收集结果: results_cnn_gru_EWS_15min

  ✓ Config base_10pct 完成

───────────